# DistilBERT — UNIFIED fake detector (CG + human deception), Colab GPU

Fine-tunes `distilbert-base-uncased` on **`combined_fake_reviews.csv`** so one model flags a review fake if it is **either computer-generated (CG) OR human-deceptive**. The human rows are ~2.5% of fakes, so human-origin TRAIN rows are **oversampled 15×** (val/test untouched). Headline metric = **per-fake-type recall** on test: does it catch CG fakes AND human fakes while sparing genuine reviews?

Tracks per-epoch **train loss / val loss / val accuracy / val macro-F1** and plots the training curves (like the LSTM/Keras notebooks).

**Setup:** GPU runtime. Upload `data/combined_fake_reviews.csv` to your **Google Drive root**, then run all (cell 1 mounts Drive).

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
CSV = '/content/drive/MyDrive/combined_fake_reviews.csv'

In [ ]:
import os
os.environ.setdefault('USE_TF', '0'); os.environ.setdefault('USE_TORCH', '1')
import numpy as np, pandas as pd, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, accuracy_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME, MAX_LEN, BATCH, LR, MAX_EPOCHS, PATIENCE, HUMAN_OVERSAMPLE = 'distilbert-base-uncased', 256, 32, 2e-5, 4, 2, 15
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device:', dev)

df = pd.read_csv(CSV); df['text'] = df['text'].astype(str)
tr, va, te = (df[df.split==s].reset_index(drop=True) for s in ('train','val','test'))
hum = tr[tr.origin=='human']
tr = pd.concat([tr] + [hum]*(HUMAN_OVERSAMPLE-1), ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'train {len(tr):,} (after {HUMAN_OVERSAMPLE}x human oversample) / val {len(va):,} / test {len(te):,}')
print('test fake_type:', dict(te.fake_type.value_counts()))

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(dev)
def enc(texts):
    e = tok(list(texts), truncation=True, padding='max_length', max_length=MAX_LEN, return_tensors='pt')
    return e['input_ids'], e['attention_mask']
def loader(d, shuffle, bs=BATCH):
    ids, am = enc(d.text); y = torch.tensor(d.label.values)
    return DataLoader(TensorDataset(ids, am, y), batch_size=bs, shuffle=shuffle)
tr_dl, va_dl, te_dl = loader(tr, True), loader(va, False), loader(te, False)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

@torch.no_grad()
def evaluate(dl):
    model.eval(); yt, yp, loss_sum, n = [], [], 0.0, 0
    for ids, am, y in dl:
        logit = model(input_ids=ids.to(dev), attention_mask=am.to(dev)).logits
        loss_sum += F.cross_entropy(logit, y.to(dev), reduction='sum').item(); n += len(y)
        yp += logit.argmax(1).cpu().tolist(); yt += y.tolist()
    return f1_score(yt,yp,average='macro',zero_division=0), accuracy_score(yt,yp), loss_sum/n, np.array(yp)

history = {'train_loss':[], 'val_loss':[], 'val_acc':[], 'val_macro_f1':[]}
best, best_state, best_epoch, bad = -1, None, 0, 0
for ep in range(1, MAX_EPOCHS+1):
    model.train(); tl, nb = 0.0, 0
    for ids, am, y in tr_dl:
        opt.zero_grad()
        loss = F.cross_entropy(model(input_ids=ids.to(dev), attention_mask=am.to(dev)).logits, y.to(dev))
        loss.backward(); opt.step(); tl += loss.item(); nb += 1
    vmf1, vacc, vloss, _ = evaluate(va_dl)
    history['train_loss'].append(tl/nb); history['val_loss'].append(vloss)
    history['val_acc'].append(vacc); history['val_macro_f1'].append(vmf1)
    print(f'epoch {ep}: train loss {tl/nb:.4f} | val loss {vloss:.4f} | val acc {vacc:.1%} | val macroF1 {vmf1:.1%}')
    if vmf1 > best: best, best_state, best_epoch, bad = vmf1, {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}, ep, 0
    else: bad += 1
    if bad >= PATIENCE: print(f'early stop at epoch {ep}'); break
model.load_state_dict(best_state)
print(f'restored best epoch {best_epoch} (val macro-F1 {best:.1%})')

In [ ]:
# training curves (train/val loss + val accuracy & macro-F1 over epochs)
eps = range(1, len(history['train_loss'])+1)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(eps, history['train_loss'], marker='o', label='train loss')
ax[0].plot(eps, history['val_loss'],   marker='o', label='val loss')
ax[0].axvline(best_epoch, ls='--', c='gray', label=f'best (ep {best_epoch})')
ax[0].set_title('Loss over epochs'); ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss'); ax[0].legend()
ax[1].plot(eps, history['val_acc'],      marker='o', label='val accuracy')
ax[1].plot(eps, history['val_macro_f1'], marker='o', label='val macro-F1')
ax[1].axvline(best_epoch, ls='--', c='gray', label=f'best (ep {best_epoch})')
ax[1].set_title('Validation accuracy & macro-F1'); ax[1].set_xlabel('epoch'); ax[1].set_ylim(0, 1); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
mf1, acc, _, tp = evaluate(te_dl); ft = te.fake_type.values
cg = (tp[ft=='cg_fake']==1).mean(); hu = (tp[ft=='human_fake']==1).mean(); ge = (tp[ft=='genuine']==0).mean()
print('='*64)
print(f'UNIFIED DistilBERT — TEST macro-F1 {mf1:.1%} | accuracy {acc:.1%}')
print(f'  CG-fake recall {cg:.1%} | human-fake recall {hu:.1%} | genuine specificity {ge:.1%}')
print('='*64)
print('(LSTM reference: macro-F1 ~98% | CG 97.7% / human 83.5% / genuine 97.7%)')

model.save_pretrained('combined_fake_distilbert'); tok.save_pretrained('combined_fake_distilbert')
import shutil; shutil.make_archive('combined_fake_distilbert', 'zip', 'combined_fake_distilbert')
from google.colab import files; files.download('combined_fake_distilbert.zip')